In [ ]:
# Execute from this cell to load the data
import pickle as pkl
import pandas as pd
data = pkl.load(open("data/mc3_binom.pkl", "rb"))
data = data.reset_index(drop=True)
data["Clonality"] = data["Clonality"].astype(str)
print(data["Clonality"].value_counts())

In [ ]:
# Create a summary table of clonality counts
# Count occurrences of Clonality for each Hugo_Symbol
clonal_summary = data.groupby("Hugo_Symbol")["Clonality"].value_counts().unstack(fill_value=0)

# Rename columns for clarity
clonal_summary.columns.name = None  # Remove column index name
clonal_summary = clonal_summary.rename(columns={"Clonal": "Clonal Count", "Sub-Clonal": "Sub-Clonal Count"})

In [ ]:
import statsmodels.api as sm

# Define the predictor and response variables
x = clonal_summary["Clonal Count"]
y = clonal_summary["Sub-Clonal Count"]

# Add a constant to the predictor variable (intercept term)
x_with_const = sm.add_constant(x)

# Fit the Poisson regression model
poisson_model = sm.GLM(y, x_with_const, family=sm.families.Poisson()).fit()

# Print the summary of the model
print(poisson_model.summary())

In [ ]:
# calculate the mean of the response variable by the predictor variable
group_means = []
xlim = 500
step = 1
for i in range(0, xlim, step):
    group_means.append(clonal_summary[(x >= i) & (x < i + step)]["Sub-Clonal Count"].mean())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm  # Ensure statsmodels is imported

# Generate values for prediction
x_plot = np.linspace(0, 400, 400)  # Generate 400 values between 0 and 400
step5_lim = 400  # Limit for x_step5
x_step5 = np.arange(0, step5_lim, 5)  # Generate values in steps of 5

# Create a design matrix for prediction (add an intercept)
x_pred = sm.add_constant(x_plot)  # Ensures it matches the Poisson model structure

# Predict the Sub-Clonal Count using the Poisson regression model
predicted_counts = poisson_model.predict(x_pred)  # Use the corrected matrix

# Create the Hexbin Plot (R-style)
fig, ax = plt.subplots(figsize=(8, 7))
hb = ax.hexbin(
    clonal_summary["Clonal Count"], clonal_summary["Sub-Clonal Count"], 
    gridsize=300, cmap="viridis", bins="log", mincnt=1  # Use log scale for density
)

# Add the Group Means
ax.plot(x_step5, group_means[0:step5_lim//5], color="red", linestyle="--", linewidth=2, label="Group Mean")

# Add the Poisson Regression Line
ax.plot(x_plot, predicted_counts, color="red", linestyle="-", linewidth=2, label="Poisson Regression")

# Improve Labels and Title (ggplot2-like styling)
ax.set_xlim([0, xlim])  # Adjust these values based on your data
ax.set_ylim([0, 2000])  # Adjust these values based on your data
ax.set_xlabel("Clonal Mutations", fontsize=10, fontweight="bold")
ax.set_ylabel("Subclonal Mutations", fontsize=10, fontweight="bold")
ax.set_title("Clonal vs. Subclonal Mutations with Poisson Regression", fontsize=14, fontweight="bold")

# Add a Colorbar for Density
cb = fig.colorbar(hb, ax=ax)
cb.set_label("Log Mutation Density", fontsize=12)

ax.legend()  # Show the legend

# Show the Plot
plt.show()